# Governança, reuso e documentação de bases

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_02/04_governanca_reuso_e_documentacao_de_bases.ipynb)

## Uma base bem documentada é automaticamente justa e reutilizável?

Uma base pode ter identificadores persistentes, metadados completos e
formato aberto e, ainda assim, ter sido disponibilizada sem participação
das comunidades relacionadas aos dados. Também pode ter acesso restrito
por razões justificáveis e continuar bem documentada e reutilizável sob
condições explícitas.

Este notebook aproxima três lentes complementares: os princípios FAIR,
os princípios CARE e a proposta de *Datasheets for Datasets*.

![Três lentes complementares examinam uma base: FAIR focaliza encontrabilidade e reuso técnico, CARE focaliza pessoas, autoridade e responsabilidade, e datasheets documentam o ciclo de vida.](imagens/04_fair_care_datasheets.svg)

**Problema orientador:** reutilizável para quem, com qual finalidade, sob
qual autoridade e com que documentação?

Ao final, você deverá ser capaz de:

1. explicar o que FAIR procura tornar possível;
2. situar CARE na governança de dados indígenas;
3. distinguir abertura, acessibilidade e autoridade;
4. adaptar perguntas de *datasheets* a uma base de Humanidades Digitais;
5. produzir uma ficha de governança e documentação para seu projeto.

Partiremos da documentação construída no Notebook 03. Primeiro
distinguiremos as perguntas de cada lente; depois as aplicaremos a
evidências, casos e decisões do projeto.

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_02'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

## 1. Três lentes, três perguntas

| Lente | Pergunta central | Unidade de atenção | O que não garante sozinha |
|---|---|---|---|
| FAIR | Os objetos digitais podem ser encontrados, acessados, combinados e reutilizados? | dados, metadados, identificadores, protocolos e vocabulários | justiça, autorização comunitária ou ausência de dano |
| CARE | Para quem os dados geram benefício, quem exerce autoridade e quais responsabilidades existem? | povos indígenas, relações, direitos, interesses e finalidades | implementação técnica ou documentação completa |
| *Datasheets* | Como e por que a base foi criada, processada, distribuída e mantida? | ciclo de vida, produtores, consumidores e pessoas afetadas | aprovação ética, qualidade absoluta ou uso futuro sem risco |

As três lentes não formam uma escala e não produzem automaticamente um
selo de qualidade. Elas tornam perguntas diferentes visíveis. Uma decisão
defensável precisa registrar tanto as evidências quanto aquilo que ainda
depende de consulta, negociação ou avaliação institucional.

A primeira lente examinada será FAIR, porque ela permite retomar de forma
direta os identificadores, metadados, formatos e condições de acesso já
documentados.

## 2. Princípios FAIR

Wilkinson et al. (2016) formularam princípios para ampliar a reutilização
de objetos digitais por pessoas e máquinas. FAIR é um acrônimo:

| Princípio | Sentido | Evidências possíveis em uma base histórica |
|---|---|---|
| **F — Findable / Encontrável** | dados e metadados possuem identificadores e podem ser localizados | identificador persistente, descrição rica e mecanismo de busca |
| **A — Accessible / Acessível** | existe um protocolo explícito para recuperar dados ou metadados | endereço documentado, protocolo padronizado e condições de acesso |
| **I — Interoperable / Interoperável** | dados e metadados podem relacionar-se a outros sistemas | formatos documentados, vocabulários e relações qualificadas |
| **R — Reusable / Reutilizável** | contexto, proveniência e condições permitem avaliar novos usos | licença ou termos, proveniência, padrões do campo e limitações |

O princípio da acessibilidade exige uma distinção importante antes da
auditoria: descrever como obter um objeto não significa torná-lo aberto a
qualquer pessoa ou finalidade.

### Acessível não significa necessariamente aberto

FAIR não exige que todo arquivo seja oferecido irrestritamente. Dados
sensíveis ou protegidos podem exigir autenticação e autorização. Nesse
caso, os metadados e o procedimento de solicitação devem permanecer tão
claros quanto possível. A pergunta não é somente “consigo baixar?”, mas
“as condições de acesso estão descritas e podem ser aplicadas?”.

FAIR também não é um padrão técnico único nem uma propriedade binária. Os
princípios orientam decisões; sua aplicação depende do domínio, da
infraestrutura e do objeto digital considerado.

Esses princípios ganham sentido quando associados a evidências verificáveis.
O experimento a seguir transforma algumas afirmações sobre a base em
perguntas auditáveis, sem reduzir FAIR a uma nota.

## 3. Experimento guiado — quais evidências podem ser auditadas?

O código abaixo não decide se a base “é FAIR”. Ele verifica apenas sinais
observáveis no pacote didático: identificadores, documentação dos campos,
condições de acesso, formato, proveniência e licença. Antes de executar,
preveja quais verificações falharão.

In [ ]:
from pathlib import Path
import json
import pandas as pd

pasta_dados = Path("dados")
catalogo = pd.read_csv(pasta_dados / "catalogo_fontes.csv")
dicionario = pd.read_csv(pasta_dados / "dicionario_dados.csv")
with (pasta_dados / "proveniencia_catalogo.json").open(encoding="utf-8") as arquivo:
    proveniencia = json.load(arquivo)

ids_validos = catalogo["id_fonte"].notna().all() and catalogo["id_fonte"].is_unique
campos_documentados = set(catalogo.columns) <= set(dicionario["campo"])
acesso_descrito = catalogo["condicao_acesso"].notna().all()

auditoria = pd.DataFrame([
    {"princípio": "F", "evidência examinada": "identificadores únicos e não vazios", "presente": ids_validos},
    {"princípio": "F", "evidência examinada": "campos do catálogo documentados", "presente": campos_documentados},
    {"princípio": "A", "evidência examinada": "condição de acesso por registro", "presente": acesso_descrito},
    {"princípio": "A", "evidência examinada": "protocolo de solicitação ou recuperação", "presente": "protocolo_acesso" in proveniencia},
    {"princípio": "I", "evidência examinada": "formato estruturado e dicionário de dados", "presente": dicionario.shape[0] > 0},
    {"princípio": "R", "evidência examinada": "origem e transformações registradas", "presente": bool(proveniencia.get("origem")) and bool(proveniencia.get("transformacoes"))},
    {"princípio": "R", "evidência examinada": "licença ou termos de reutilização", "presente": "licenca" in proveniencia},
])
auditoria

### Interpretação da auditoria

1. Qual ausência impede compreender como solicitar acesso?
2. Por que `condicao_acesso` não equivale a uma licença de reutilização?
3. Que aspectos de interoperabilidade não foram examinados pelo código?
4. Por que não devemos somar a coluna `presente` para produzir uma “nota
   FAIR” definitiva?

**Minha interpretação:** Escreva aqui.

O resultado é uma lista de evidências e lacunas. Ele não avalia a
qualidade semântica dos metadados, a estabilidade futura dos
identificadores, a adequação dos vocabulários nem a legitimidade das
condições de acesso.

A auditoria alcança infraestrutura e documentação, mas não determina quem
possui autoridade legítima nem quem recebe benefícios ou suporta riscos.
Essa insuficiência motiva a passagem aos princípios CARE.

## 4. Princípios CARE e soberania de dados indígenas

Os princípios CARE foram desenvolvidos em consulta com povos indígenas e
redes de soberania de dados indígenas. Carroll et al. (2020) os apresentam
como orientação centrada em pessoas e propósitos, complementar à ênfase
de FAIR nos dados e na infraestrutura.

| Princípio | Questões para análise situada |
|---|---|
| **C — Collective Benefit / Benefício coletivo** | A iniciativa responde a prioridades definidas pelos povos envolvidos? Que benefícios retornam às comunidades? |
| **A — Authority to Control / Autoridade para controlar** | Quem tem direito e poder efetivo para decidir coleta, acesso, uso e reutilização? |
| **R — Responsibility / Responsabilidade** | Que relações, capacidades e deveres de cuidado devem ser mantidos ao longo do projeto? |
| **E — Ethics / Ética** | Como minimizar danos, promover justiça e permitir que os próprios povos avaliem riscos e usos futuros? |

CARE não deve ser separado de sua origem e convertido em uma lista ética
genérica. Quando o projeto envolver dados, conhecimentos, territórios ou
patrimônio de povos indígenas, a aplicação exige participação e
autoridade reais; responder individualmente a um formulário não demonstra
conformidade. Em outros contextos, CARE pode provocar perguntas úteis
sobre poder e benefício, mas o projeto não deve alegar adesão aos
princípios sem justificar seu âmbito e sua governança.

Com as duas lentes definidas, podemos observar como abertura técnica e
autoridade situada podem apontar para decisões diferentes. As tensões não
são falhas do método: são questões que o projeto precisa explicitar.

## 5. FAIR e CARE podem produzir tensões produtivas

Compare os casos fictícios:

| Caso | Leitura por FAIR | Leitura por CARE |
|---|---|---|
| Um repositório publica imagens, metadados e API aberta de uma coleção indígena sem participação do povo relacionado | os objetos podem ser encontráveis, acessíveis e interoperáveis | abertura técnica não responde quem autorizou, quem se beneficia nem quais usos causam dano |
| Uma comunidade mantém certos registros restritos, mas publica metadados e um procedimento de acesso sob sua autoridade | a restrição não impede necessariamente FAIR, se o protocolo estiver documentado | o controle de acesso pode expressar autoridade, responsabilidade e ética |

Os casos tornam a tensão concreta. A discussão em duplas pede que você
transforme a comparação em uma decisão condicionada por evidências e pela
autoridade das pessoas envolvidas.

### Discussão em duplas — abrir, restringir ou negociar?

**Dinâmica:** em 5 minutos, cada estudante escolhe um caso e formula uma
decisão. Em 10 minutos, a dupla compara as decisões usando ao menos uma
evidência FAIR e duas questões CARE. Em 10 minutos, as duplas apresentam
uma condição que faria sua decisão mudar.

Não procure uma regra universal. Identifique quem pode decidir, quais
informações faltam e quais danos ou benefícios cada forma de acesso pode
produzir.

A discussão produz decisões e condições, mas elas precisam permanecer
registradas para futuros responsáveis e usuários. A proposta de
*Datasheets for Datasets* oferece uma estrutura para essa documentação ao
longo do ciclo de vida.

## 6. *Datasheets for Datasets*

Gebru et al. (2021) propõem que uma base seja acompanhada por um documento
estruturado que ajude produtores e usuários a refletir sobre sua criação e
seus usos. As perguntas acompanham aproximadamente o ciclo de vida:

| Parte | O que documentar em uma adaptação para Humanidades Digitais |
|---|---|
| motivação | pergunta, finalidade, equipe, instituições e financiamento |
| composição | unidade, população, corpus, campos, lacunas e pessoas ou grupos representados |
| coleta | fontes, custodiantes, período, seleção, consentimento ou autorizações aplicáveis |
| processamento | transcrição, OCR, classificação, exclusões, normalização e versões preservadas |
| usos | análises previstas, usos inadequados, riscos e limites das inferências |
| distribuição | forma de acesso, licença, termos, restrições e documentação fornecida |
| manutenção | responsável, contato, correções, novas versões, retenção e descontinuação |

A proposta nasceu no contexto de aprendizado de máquina. Aqui ela será
adaptada criticamente para catálogos, corpus e coleções digitais. Ela não
substitui descrição arquivística, plano de gestão, avaliação ética,
parecer jurídico ou consulta às comunidades relacionadas aos dados.

Os autores também alertam que a criação da ficha não deve ser totalmente
automatizada: seu valor está na reflexão sobre escolhas, riscos e
responsabilidades, não apenas na extração de propriedades técnicas.

Agora as três lentes podem ser reunidas. A atividade transforma evidências
FAIR, questões CARE e documentação do ciclo de vida em uma única ficha
destinada ao protocolo final.

## 7. Atividade integrada — ficha de governança e documentação

**Produto:** uma ficha de uma a duas páginas que será incorporada ao
protocolo da oficina.

**Dinâmica:** trabalhe individualmente por 20 minutos; troque a ficha com
uma dupla por 15 minutos; revise-a por 10 minutos a partir do parecer.

### Parte A — evidências FAIR

Use a auditoria como ponto de partida. Para cada princípio, registre uma
evidência existente, uma lacuna e uma ação possível. Não atribua nota
numérica.

| Princípio | Evidência | Lacuna | Ação ou decisão |
|---|---|---|---|
| F | Escreva aqui | Escreva aqui | Escreva aqui |
| A | Escreva aqui | Escreva aqui | Escreva aqui |
| I | Escreva aqui | Escreva aqui | Escreva aqui |
| R | Escreva aqui | Escreva aqui | Escreva aqui |

As evidências técnicas delimitam o que já está documentado. Em seguida,
examine quem pode decidir, beneficiar-se ou ser afetado pelas formas de
coleta e uso.

### Parte B — governança e CARE

**Os dados envolvem povos indígenas ou conhecimentos, territórios e
patrimônio a eles relacionados? Que consulta fundamenta a resposta?**
Escreva aqui.

**Quem se beneficia e quem pode autorizar ou recusar usos?** Escreva aqui.

**Que responsabilidades continuarão após a coleta ou publicação?**
Escreva aqui.

**Que riscos e usos futuros precisam ser avaliados pelas pessoas ou
comunidades afetadas?** Escreva aqui.

Essas decisões de governança precisam ser comunicadas junto à motivação,
composição, distribuição e manutenção da base. Use-as para preencher o
minidatasheet.

### Parte C — minidatasheet

**Motivação e responsáveis:** Escreva aqui.

**Composição, cobertura e lacunas:** Escreva aqui.

**Coleta e processamento previstos:** Escreva aqui.

**Usos recomendados e usos inadequados:** Escreva aqui.

**Distribuição, acesso e licença:** Escreva aqui.

**Manutenção, contato e versionamento:** Escreva aqui.

A ficha ainda expressa apenas a perspectiva de quem a escreveu. A revisão
por pares procura inconsistências e ausências antes de sua incorporação ao
protocolo da oficina.

### Parte D — parecer da dupla e revisão

O colega deve localizar: uma afirmação sem evidência; uma condição de
acesso confundida com licença; uma pessoa ou coletividade afetada ainda
ausente; e um uso futuro que precisa ser limitado.

**Parecer recebido:** Escreva aqui.

**Mudança realizada e justificativa:** Escreva aqui.

Depois da revisão, a síntese abaixo recupera a função específica de cada
lente e explicita o produto que seguirá para o Notebook 05.

## 8. Síntese e leituras

- FAIR orienta a encontrabilidade, o acesso, a interoperabilidade e o
  reuso de objetos digitais, inclusive por máquinas.
- CARE recoloca povos indígenas, autoridade, relações, benefício e ética
  no centro da governança de dados.
- *Datasheets* organizam a documentação reflexiva do ciclo de vida.
- documentação não transforma uma decisão injusta em decisão justa;
  restrição justificada não torna automaticamente uma base mal gerida.

**Leituras essenciais:** Wilkinson et al. (2016), Carroll et al. (2020) e
Gebru et al. (2021). Dados bibliográficos e links: `referencias.md`.

Leve a ficha revisada para o Notebook 05. Ela sustentará as decisões sobre
documentação, acesso, governança, ética e usos do protocolo final.